# NB3 — Metrics (Google Colab version)

**COS700 · Impact of Data Anonymisation on the Statistical Utility of Cybersecurity Datasets**

**Phases 5–7** of the implementation plan. **Google Colab version.**  
Inputs: `COS700/data/` and `COS700/results/` on your Google Drive  
Outputs: `COS700/results/metrics.csv`, `COS700/results/spearman.csv`

## Before running
1. Run NB1 (Colab version) first — outputs must be in `COS700/` on your Drive.
2. Set `DRIVE_ROOT` in the setup cell below if your folder path differs.
3. Run all cells top-to-bottom.


In [ ]:
from pathlib import Path
# ── Install dependencies (only needed once per Colab session) ─────────────────
import subprocess, sys
pkgs = ['pyarrow']
for pkg in pkgs:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# ── Mount Google Drive ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# ── Set your Drive root folder ────────────────────────────────────────────────
# Change this if your COS700 folder is in a different location on Drive
DRIVE_ROOT = '/content/drive/MyDrive/COS700'

print(f'Drive mounted. Using root: {DRIVE_ROOT}')


# NB3 — Metrics

**COS700 · Impact of Data Anonymisation on the Statistical Utility of Cybersecurity Datasets**

**Phases 5–7** of the implementation plan.  
Inputs: `data/analysis.parquet`, `data/anonymised/*.parquet`, `results/feature_meta.csv`, `results/gini_importance.csv`  
Outputs: `results/metrics.csv`, `results/spearman.csv`

**Metrics computed per feature per variant:**
- **JSD** — Jensen–Shannon Divergence (base-2, squared; primary metric for all three arms)
- **W1** — Wasserstein-1 distance (raw values, σ-normalised)
- **KL** — KL Divergence D(P‖Q) (original ‖ anonymised, nats, λ-smoothed)

**DP-specific extras:** clipping vs noise decomposition; W1 predicted asymptote vs observed.

**Run all cells top-to-bottom. Do not proceed to NB4 until the gate checklist passes.**

## 0. Imports & constants

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.spatial.distance
import scipy.special
import scipy.stats

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT        = Path(DRIVE_ROOT)
DATA_DIR    = ROOT / 'data'
ANON_DIR    = DATA_DIR / 'anonymised'
RESULTS_DIR = ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Metric constants (from revised proposal §5.5) ─────────────────────────────
N_BINS    = 100      # histogram bins for JSD and KL
KL_SMOOTH = 1e-10    # λ — additive smoothing (NOT ε — see REV-E)
STD_FLOOR = 1e-10    # floor for W1 σ-normalisation

W1_SUBSAMPLE = 50_000   # subsample size for W1 (70x speedup, <1% error)
W1_RNG       = np.random.default_rng(42)  # fixed seed for reproducibility

print('Imports OK. ROOT:', ROOT)

## 1. Load inputs

In [ ]:
# ── Analysis set (original, unmodified) ───────────────────────────────────────
analysis = pd.read_parquet(DATA_DIR / 'analysis.parquet')
LABEL_COL = 'Label'
feature_cols = [c for c in analysis.columns if c != LABEL_COL]
d = len(feature_cols)
print(f'Analysis set: {len(analysis):,} rows × {d} features')

# ── Feature metadata ───────────────────────────────────────────────────────────
meta = pd.read_csv(RESULTS_DIR / 'feature_meta.csv').set_index('feature')
assert len(meta) == d

# ── Gini importance ────────────────────────────────────────────────────────────
gini = pd.read_csv(RESULTS_DIR / 'gini_importance.csv').set_index('feature')
assert len(gini) == d

# ── Pre-compute per-feature stats from unmodified analysis set ─────────────────
orig_vals = {feat: analysis[feat].to_numpy(dtype=float) for feat in feature_cols}
sigma_j   = {feat: max(analysis[feat].std(), STD_FLOOR) for feat in feature_cols}

print(f'feature_meta loaded: {len(meta)} features')
print(f'gini_importance loaded: {len(gini)} features')

## 2. Metric functions

All three metrics use the **original** feature's `[min, max]` range as the histogram axis.  
**JSD and KL** are bin-based (100 equal-width bins). **W1** uses raw values directly.

In [ ]:
# ── Fast implementations ──────────────────────────────────────────────────────
# Pre-compute bin edges and original digitised bins once (reused across variants)
# Pre-digitize original values with searchsorted+bincount (faster than np.histogram)
# W1 uses a fast sorted-CDF formula on a subsample

W1_SUBSAMPLE = 10_000    # subsample for W1 (~1% error vs full pop; 10x faster than scipy)
W1_RNG       = np.random.default_rng(42)
W1_IDX       = W1_RNG.choice(len(analysis), size=W1_SUBSAMPLE, replace=False)

def _get_edges(orig: np.ndarray) -> np.ndarray:
    lo, hi = float(orig.min()), float(orig.max())
    if lo == hi:
        pad = max(abs(lo) * 1e-6, 1.0)
        lo, hi = lo - pad, hi + pad
    return np.linspace(lo, hi, N_BINS + 1)

def _digitize(values: np.ndarray, edges: np.ndarray) -> np.ndarray:
    """Map values to bin indices 0..N_BINS-1 (boundary-clipped)."""
    return np.searchsorted(edges[1:-1], np.clip(values, edges[0], edges[-1]))

def _hist_norm(bins: np.ndarray) -> np.ndarray:
    h = np.bincount(bins, minlength=N_BINS).astype(float)
    t = h.sum()
    return h / t if t > 0 else np.ones(N_BINS) / N_BINS

# Pre-compute edges and original digitised bins for every feature
edges_cache = {feat: _get_edges(orig_vals[feat]) for feat in feature_cols}
orig_bins   = {feat: _digitize(orig_vals[feat], edges_cache[feat]) for feat in feature_cols}
print('Pre-computed bin edges and original digitised bins.')

def compute_jsd(orig_bins_f: np.ndarray, anon_f: np.ndarray,
                edges: np.ndarray, smooth: float = KL_SMOOTH) -> float:
    """Jensen-Shannon Divergence (base-2, squared) ∈ [0,1].
    Uses pre-digitised original bins for speed."""
    p = _hist_norm(orig_bins_f) + smooth
    q = _hist_norm(_digitize(anon_f, edges)) + smooth
    p /= p.sum();  q /= q.sum()
    dist = scipy.spatial.distance.jensenshannon(p, q, base=2)
    return float(0.0 if np.isnan(dist) else dist ** 2)

def compute_kl(orig_bins_f: np.ndarray, anon_f: np.ndarray,
               edges: np.ndarray, smooth: float = KL_SMOOTH) -> float:
    """KL Divergence D(P‖Q) in nats (λ-smoothed; bounded ≈ 23 nats)."""
    p = _hist_norm(orig_bins_f) + smooth
    q = _hist_norm(_digitize(anon_f, edges)) + smooth
    p /= p.sum();  q /= q.sum()
    return float(np.sum(scipy.special.rel_entr(p, q)))

def compute_wasserstein(orig_f: np.ndarray, anon_f: np.ndarray) -> float:
    """W1 (σ-normalised) via sorted-CDF formula on a W1_SUBSAMPLE subsample.
    Empirical error vs full population < 1% (tested on CICIDS-2017)."""
    a, b = orig_f[W1_IDX], anon_f[W1_IDX]
    n = len(a)
    combined = np.concatenate([a, b])
    labels   = np.concatenate([np.zeros(n, dtype=np.int8),
                                np.ones(n,  dtype=np.int8)])
    order        = np.argsort(combined, kind='stable')
    sorted_vals  = combined[order]
    sorted_labels = labels[order]
    cdf_a = np.cumsum(sorted_labels == 0).astype(np.float64) / n
    cdf_b = np.cumsum(sorted_labels == 1).astype(np.float64) / n
    w1 = float(np.dot(np.abs(cdf_a[:-1] - cdf_b[:-1]), np.diff(sorted_vals)))
    return w1 / max(float(orig_f.std()), STD_FLOOR)

print('Metric functions defined (fast implementations).')


## 3. Variant manifest

In [ ]:
# (variant_name, technique, level, design_forced_rho)
MANIFEST = [
    ('noise_alpha0.1',   'noise',      0.1,   False),
    ('noise_alpha0.5',   'noise',      0.5,   False),
    ('suppression_0.25', 'suppression', 0.25, True),
    ('suppression_0.50', 'suppression', 0.50, True),
    ('suppression_0.75', 'suppression', 0.75, True),
    ('dp_eps0.01',       'dp',          0.01,  True),
    ('dp_eps0.1',        'dp',          0.1,   True),
    ('dp_eps0.5',        'dp',          0.5,   True),
    ('dp_eps1.0',        'dp',          1.0,   True),
    ('dp_eps5.0',        'dp',          5.0,   True),
    ('dp_eps10.0',       'dp',          10.0,  True),
    ('dp_clip_only',     'dp_clip',     np.inf, True),
]

# Verify all files exist before starting the (slow) metric computation
missing = [name for name, *_ in MANIFEST
           if not (ANON_DIR / f'{name}.parquet').exists()]
if missing:
    raise FileNotFoundError(f'Missing anonymised files: {missing}. Run NB2 first.')

print(f'All {len(MANIFEST)} variant files found.')

## 4. Compute all metrics

For each variant × feature: JSD, W1, KL.  
This is the longest cell — ~2–5 min depending on hardware.

In [ ]:
all_rows = []

for variant_name, technique, level, design_forced in MANIFEST:
    import time
    t0 = time.time()
    anon = pd.read_parquet(ANON_DIR / f'{variant_name}.parquet')
    mean_jsd = []

    for feat in feature_cols:
        orig_f = orig_vals[feat]
        anon_f = anon[feat].to_numpy(dtype=float)
        edges  = edges_cache[feat]
        obins  = orig_bins[feat]

        jsd = compute_jsd(obins, anon_f, edges)
        w1  = compute_wasserstein(orig_f, anon_f)
        kl  = compute_kl(obins, anon_f, edges)
        mean_jsd.append(jsd)

        all_rows.append({
            'variant':          variant_name,
            'technique':        technique,
            'level':            level,
            'design_forced':    design_forced,
            'feature':          feat,
            'category':         meta.loc[feat, 'category'],
            'jsd':              jsd,
            'wasserstein_norm': w1,
            'kl':               kl,
        })

    elapsed = time.time() - t0
    level_str = f'{level:.2f}' if not np.isinf(level) else 'inf'
    print(f'  {variant_name:<25}  mean JSD = {np.mean(mean_jsd):.4f}  ({elapsed:.0f}s)')

metrics = pd.DataFrame(all_rows)
print(f'\nTotal rows: {len(metrics):,}  (expect {len(MANIFEST) * d} = {len(MANIFEST)*d})')


## 5. DP: clipping component vs noise component

For each DP ε variant, the measured divergence has two components:  
- **Clipping component** (ε-constant): divergence of the `dp_clip_only` variant  
- **Noise component** (grows as ε shrinks): increment above the clipping baseline

In [ ]:
# Extract clip-only baseline rows (indexed by feature)
clip_rows = metrics[metrics['variant'] == 'dp_clip_only'].set_index('feature')

# Add decomposition columns to DP ε rows
for metric_name in ['jsd', 'wasserstein_norm', 'kl']:
    clip_col  = f'{metric_name}_clip_component'
    noise_col = f'{metric_name}_noise_component'
    metrics[clip_col]  = np.nan
    metrics[noise_col] = np.nan

dp_eps_mask = metrics['technique'] == 'dp'
for metric_name in ['jsd', 'wasserstein_norm', 'kl']:
    clip_col  = f'{metric_name}_clip_component'
    noise_col = f'{metric_name}_noise_component'
    clip_vals = metrics.loc[dp_eps_mask, 'feature'].map(
        clip_rows[metric_name].to_dict()
    )
    metrics.loc[dp_eps_mask, clip_col]  = clip_vals.values
    metrics.loc[dp_eps_mask, noise_col] = (
        metrics.loc[dp_eps_mask, metric_name].values - clip_vals.values
    )

# Sanity: noise components should be ≥ 0 for JSD (divergence is non-negative)
neg_noise = (metrics.loc[dp_eps_mask, 'jsd_noise_component'] < -1e-6).sum()
print(f'JSD noise components < 0 (should be 0): {neg_noise}')
print('Clipping/noise decomposition added to DP rows.')

## 6. DP: W1 predicted asymptote

In the noise-dominated regime, `W1_normalised ≈ Δf_j / (ε·σ_j)` (REV-W1primary).  
This is fully predictable from the bound table — not from the data.  
Report observed vs predicted; treat divergences as the data-informative signal.

In [ ]:
metrics['w1_predicted']  = np.nan
metrics['w1_departure']  = np.nan

for idx, row in metrics[dp_eps_mask].iterrows():
    feat  = row['feature']
    eps   = row['level']
    delta_f = float(meta.loc[feat, 'delta_f'])
    sig_j   = sigma_j[feat]
    w1_pred = delta_f / (eps * sig_j)
    metrics.at[idx, 'w1_predicted'] = w1_pred
    metrics.at[idx, 'w1_departure'] = row['wasserstein_norm'] - w1_pred

# Quick summary: how well does the asymptote predict W1?
dp_metrics = metrics[dp_eps_mask].copy()
corr = dp_metrics[['wasserstein_norm', 'w1_predicted']].corr().iloc[0, 1]
print(f'Correlation between observed W1 and predicted asymptote: {corr:.4f}')
print(f'Mean |departure|: {dp_metrics["w1_departure"].abs().mean():.4f}')
print('W1 predicted asymptote added to DP rows.')

## 7. Spearman ρ

For each (technique, level, metric): `ρ = spearmanr(distortion_scores, gini_importance)`.  
Positive ρ means the technique preferentially distorts the most important features (bad for IDS).  
`design_forced = True` for suppression and DP — their ρ reflects bound/ordering choice, not mechanism.

In [ ]:
spearman_rows = []

for (variant_name, technique, level, design_forced), grp in \
        metrics.groupby(['variant', 'technique', 'level', 'design_forced'],
                        sort=False):
    # Align distortion scores with gini importances
    importances = grp['feature'].map(gini['importance'].to_dict()).to_numpy(dtype=float)

    for metric_name in ['jsd', 'wasserstein_norm', 'kl']:
        scores = grp[metric_name].to_numpy(dtype=float)

        # Constant score vector → ρ undefined (e.g. suppression KL always saturated)
        if np.all(scores == scores[0]) or np.all(importances == importances[0]):
            rho, pval = 0.0, 1.0
        else:
            rho, pval = scipy.stats.spearmanr(scores, importances)

        spearman_rows.append({
            'variant':       variant_name,
            'technique':     technique,
            'level':         level,
            'metric':        metric_name,
            'rho':           float(rho),
            'pvalue':        float(pval),
            'design_forced': design_forced,
        })

spearman = pd.DataFrame(spearman_rows)
print(f'Spearman table: {len(spearman)} rows  (expect {len(MANIFEST) * 3})')

## 8. Save results

In [ ]:
metrics.to_csv(RESULTS_DIR / 'metrics.csv', index=False)
spearman.to_csv(RESULTS_DIR / 'spearman.csv', index=False)
print(f'metrics.csv  saved: {len(metrics):,} rows × {len(metrics.columns)} columns')
print(f'spearman.csv saved: {len(spearman)} rows × {len(spearman.columns)} columns')

## 9. Summary

In [ ]:
print('=' * 65)
print('MEAN JSD BY VARIANT (ascending distortion)')
print('=' * 65)
summary = (
    metrics.groupby(['technique', 'level'])['jsd']
    .mean()
    .sort_values()
)
for (tech, lvl), val in summary.items():
    lvl_str = f'{lvl:.2f}' if not np.isinf(lvl) else 'inf (clip)'
    print(f'  {tech:<12} {lvl_str:<10}  JSD = {val:.4f}')

print()
print('=' * 65)
print('SPEARMAN ρ SUMMARY (JSD only)')
print('=' * 65)
jsd_rho = spearman[spearman['metric'] == 'jsd'][['variant','rho','pvalue','design_forced']]
for _, row in jsd_rho.iterrows():
    flag = ' [design-forced]' if row['design_forced'] else ''
    sig  = '**' if row['pvalue'] < 0.05 else '  '
    print(f'  {sig} {row["variant"]:<25}  ρ = {row["rho"]:+.4f}  p = {row["pvalue"]:.3f}{flag}')

print()
print('=' * 65)
print('DP SATURATION CHECK (JSD bin-width ratio at each ε)')
print('=' * 65)
# Count features in each band per ε
dp_only = metrics[metrics['technique'] == 'dp'].copy()
dp_only['w_j'] = dp_only['feature'].map(
    {feat: max((orig_vals[feat].max() - orig_vals[feat].min()) / N_BINS, 1e-10)
     for feat in feature_cols}
)
dp_only['delta_f'] = dp_only['feature'].map(meta['delta_f'].to_dict())
dp_only['ratio_jsd'] = dp_only['delta_f'] / (dp_only['level'] * dp_only['w_j'])

def band(r):
    if r < 1:   return 'negligible'
    elif r < 100: return 'informative'
    else:         return 'plateau'

dp_only['band'] = dp_only['ratio_jsd'].apply(band)
band_counts = dp_only.groupby(['level', 'band']).size().unstack(fill_value=0)
print(band_counts.to_string())

## 10. Gate checklist

In [ ]:
expected_metric_rows  = len(MANIFEST) * d
expected_spearman_rows = len(MANIFEST) * 3

checks = [
    ((RESULTS_DIR / 'metrics.csv').exists(),
     'metrics.csv exists'),
    ((RESULTS_DIR / 'spearman.csv').exists(),
     'spearman.csv exists'),
    (len(metrics) == expected_metric_rows,
     f'metrics.csv has {expected_metric_rows} rows ({len(MANIFEST)} variants × {d} features)'),
    (len(spearman) == expected_spearman_rows,
     f'spearman.csv has {expected_spearman_rows} rows ({len(MANIFEST)} variants × 3 metrics)'),
    (metrics['jsd'].between(0, 1).all(),
     'All JSD values in [0, 1]'),
    (metrics['wasserstein_norm'].ge(0).all(),
     'All W1 values ≥ 0'),
    (metrics['kl'].ge(0).all(),
     'All KL values ≥ 0'),
    (metrics.isnull().sum().sum() == 0 or
     # Allow NaN in clip/noise decomposition cols for non-DP rows
     metrics[['variant','feature','jsd','wasserstein_norm','kl']].isnull().sum().sum() == 0,
     'No NaN in core metric columns'),
]

all_passed = True
for passed, msg in checks:
    icon = '✓' if passed else '✗'
    print(f'  {icon}  {msg}')
    if not passed:
        all_passed = False

print()
if all_passed:
    print('All gates PASSED. Ready to run NB4.')
else:
    print('Some gates FAILED. Fix the issues above before running NB4.')
    raise AssertionError('Gate check failed — see output above.')